# Use Case — Portfolio Heat Evaluation for the Real-Estate Agent

**Who this is for**  
Real-estate agents preparing a portfolio review with a client; portfolio managers and client advisors; secondarily REIT asset managers, private real-estate investors, and property operations leads.

**The scenario**  
You manage (or advise on) a 10-asset San Jose portfolio. The client wants to know which assets are at heat risk *today*, which represent the biggest *opportunity* if you treat them, and what those answers mean in dollars. Walking a building per day is not an option — you need a desk-first screening that gives you defensible numbers and slide-ready visuals before the meeting.

This notebook combines **your portfolio data** with **FortyGuard layers** to answer five questions:

1. **Where in the portfolio is hot, when, and by how much?**  ← 24-hour heatmap × portfolio
2. **Why are the hottest properties hot?**  ← satellite segmentation on top exposures
3. **What does the curb actually look like?**  ← street-view ground-truth on the #1 exposure
4. **What does tenant comfort look like through the day?**  ← environmental parameters
5. **What's the risk and the opportunity per asset, in dollars and tiers?**  ← composite scoring + business translation

> **Cached by default.** The notebook ships with `CACHED=True` so it runs end-to-end against the bundled sample files in `data/` — no API key needed for the demo. Set `CACHED=False` at the top of the Setup cell to run the same workflow live against any portfolio.

> **Bring your own portfolio.** Sample data ships at `data/real_estate_san_jose_portfolio_sample.csv`. Swap the path in Step 1 — as long as the columns match (`property_id`, `name`, `type`, `year_built`, `sqft`, `market_value_musd`, `latitude`, `longitude`), everything downstream works.

---

## Setup

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

import json
import random
import pandas as pd
import folium
import matplotlib.pyplot as plt
from shapely.geometry import Point, shape, mapping
from IPython.display import HTML, display

from fortyguard import FortyGuardClient
from fortyguard.samples import SAN_JOSE_POLYGON

# ── configuration ─────────────────────────────────────
STUDY_DATE         = '2024-10-02'         # matches the bundled cached files
STUDY_HOUR         = '14:00'              # design-peak snapshot for the live API
GRANULARITY_M      = 100                  # heatmap resolution
TOP_N_TO_ENRICH    = 3                    # API budget for satellite / env-params
CACHED             = True                 # set False for live API calls
BASELINE_C         = 24.0                 # comfortable ambient baseline
COOLING_KWH_PER_SF = 0.18                 # extra kWh per sf per °C above baseline
KWH_PRICE_USD      = 0.24
SLA_HI_C           = 32.0                 # tenant-comfort heat-index threshold
TEMP_C_SANITY      = (15.0, 55.0)         # F→C conversion sanity range
ASSET_TYPE_PALETTE = {
    'Office':       '#1f77b4',
    'Residential':  '#2ca02c',
    'Retail':       '#ff7f0e',
    'Mixed-Use':    '#9467bd',
    'Industrial':   '#8c564b',
}

# ── data paths ─────────────────────────────────────────
DATA            = ROOT / 'data'
PORTFOLIO_CSV   = DATA / 'real_estate_san_jose_portfolio_sample.csv'
HEATMAP_GEOJSON = DATA / 'real_state_san_jose_heatmap_sample_day_2024-10-02.geojson'
SATELLITE_JSON  = DATA / 'real_state_san_jose_satellite_segmentation_sample_day_2024-10-02.json'
STREETVIEW_JSON = DATA / 'real_state_san_jose_street_view_segmentation_sample_day_2024-10-02.json'
ENV_PARAMS_JSON = DATA / 'real_state_san_jose_env_paramaters_sample_day_2024-10-02.json'
HEAT_INTEL_PDF  = DATA / 'real_state_san_jose_heat_intelligence_sample_day_2024-10-02.pdf'
OUTPUT_CSV      = ROOT / 'outputs' / 'portfolio_evaluation.csv'
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

client = FortyGuardClient()  # used only when CACHED=False
print(f"CACHED={CACHED}  STUDY_DATE={STUDY_DATE}  TOP_N_TO_ENRICH={TOP_N_TO_ENRICH}")

CACHED=True  STUDY_DATE=2024-10-02  TOP_N_TO_ENRICH=3


---
## Step 1 — Load your portfolio

### What you are doing
Reading the portfolio CSV. Any columns may ride along — the workflow only needs `latitude` and `longitude` to do the geospatial work; the rest pass through to the final output so finance and ops see the asset IDs and values they already recognize.

### Why this matters
Starting from the operations system of record means the output carries *your* property IDs, *your* asset types, *your* square-footages — ready to paste into the client report or CRM.

In [2]:
portfolio = pd.read_csv(PORTFOLIO_CSV)
type_counts = portfolio['type'].value_counts().to_dict()
print(f"Loaded {len(portfolio)} properties, "
      f"total value ${portfolio['market_value_musd'].sum():.0f}M, "
      f"total {portfolio['sqft'].sum():,} sqft")
print(f"  asset-type mix: {type_counts}")
portfolio

Loaded 10 properties, total value $2155M, total 2,690,000 sqft
  asset-type mix: {'Office': 4, 'Residential': 3, 'Retail': 1, 'Mixed-Use': 1, 'Industrial': 1}


,property_id,name,type,year_built,sqft,market_value_musd,latitude,longitude
0,P01,Adobe Campus Tower,Office,2015,620000,450,37.2963,-121.8341
1,P02,Downtown Commons Residences,Residential,2019,340000,380,37.3410,-121.8890
2,P03,City View Office Plaza,Office,1987,450000,280,37.3370,-121.8905
3,P04,Santa Clara Street Retail,Retail,1952,45000,35,37.3378,-121.8870
4,P05,San Pedro Square Mixed-Use,Mixed-Use,2005,220000,195,37.3376,-121.8940
5,P06,Japantown Lofts,Residential,1920,95000,88,37.3445,-121.8945
6,P07,Paseo Office Center,Office,1998,320000,260,37.3362,-121.8895
7,P08,Southside Industrial,Industrial,1965,180000,62,37.3315,-121.8920
8,P09,The Fairmont Residences,Residential,2010,140000,225,37.3328,-121.8870
9,P10,Tech Corridor Offices,Office,1989,280000,180,37.3392,-121.8810


---
## Step 2 — Heat layer (online or cached)

### What you are doing
When `CACHED=True` we read the bundled 24-hour heatmap GeoJSON — 16,507 tiles, each carrying hourly temperatures `'00'..'23'` plus min/max/avg. Values in the file are in °F, so we convert and assert a sanity range. When `CACHED=False` we call `client.create_heatmap` for the design hour.

### Why this matters
A city-wide weather observation misses intra-city variation that *actually* drives cooling cost and tenant complaints. At 100 m resolution we can distinguish a building on the hot side of a block from one on the cool side. The 24-hour cached layer also unlocks **peak-hour** and **diurnal-swing** signals you cannot get from a single-hour call.

In [ ]:
import numpy as np
import textwrap
from matplotlib.colors import LinearSegmentedColormap

# 12-stop spectral ramp (cool blue → hot red) used for the summary card,
# the histogram, and the colorbar. Reused later by the M1/M2/M3 maps.
TCM_COLORS = [
    '#2983ba', '#5aa4b2', '#88c4aa', '#b3e0a6', '#d1ecb0', '#f0f9ba',
    '#fff0ae', '#fed38c', '#fdb56a', '#f3854e', '#e54f35', '#d7191c',
]
TCM_CMAP = LinearSegmentedColormap.from_list('tcm', TCM_COLORS, N=256)

def temp_color(t, lo, hi):
    """Map a temperature to one of TCM_COLORS by linear interpolation in [lo, hi]."""
    if t is None or lo is None or hi is None or hi == lo:
        return TCM_COLORS[0]
    frac = max(0.0, min(1.0, (float(t) - lo) / (hi - lo)))
    idx  = min(int(frac * len(TCM_COLORS)), len(TCM_COLORS) - 1)
    return TCM_COLORS[idx]

def show_heatmap_summary(temps, source_label):
    """Stats card + colored histogram + vertical colorbar — one figure summarizing
    the AOI temperature distribution. `temps` is the per-tile peak (°C) list."""
    temps = [t for t in temps if t is not None]
    if not temps:
        print('No tile temperatures to summarize.')
        return
    lo, hi = float(min(temps)), float(max(temps))
    mean   = float(sum(temps) / len(temps))

    wrapped_label = textwrap.fill(source_label, width=22) if source_label else ''
    n_label_lines = wrapped_label.count('\n') + 1

    # Grow the figure a bit when the source label wraps to many lines, so the
    # stats column has room for both title and the min/mean/max rows.
    fig = plt.figure(figsize=(12, 3.4 + 0.30 * max(0, n_label_lines - 1)),
                     constrained_layout=True)
    gs = fig.add_gridspec(1, 3, width_ratios=[1.4, 2.6, 0.20])

    # Stats card -----------------------------------------------------------
    ax0 = fig.add_subplot(gs[0, 0]); ax0.axis('off')
    ax0.text(0.0, 0.97, wrapped_label, transform=ax0.transAxes,
             fontsize=10.5, fontweight='bold', color='#222', va='top')
    subtitle_y = 0.97 - 0.11 * n_label_lines - 0.05
    ax0.text(0.0, subtitle_y, f"{len(temps):,} tiles",
             transform=ax0.transAxes,
             fontsize=10, color='#666', va='top')

    rows = [('min',  lo,   temp_color(lo,   lo, hi)),
            ('mean', mean, temp_color(mean, lo, hi)),
            ('max',  hi,   temp_color(hi,   lo, hi))]
    # Pack rows into the space below the subtitle so a multi-line filename
    # never crashes into the min/mean/max swatches.
    band_top    = subtitle_y - 0.10
    band_bottom = 0.05
    step        = (band_top - band_bottom) / max(len(rows) - 1, 1)
    rect_h      = min(0.14, step * 0.6)
    y = band_top
    for label, val, color in rows:
        ax0.text(0.0, y, label, transform=ax0.transAxes,
                 fontsize=10, color='#666', va='center', family='monospace')
        ax0.add_patch(plt.Rectangle((0.22, y - rect_h / 2), 0.10, rect_h,
                                    transform=ax0.transAxes,
                                    facecolor=color, edgecolor='#333', linewidth=0.6))
        ax0.text(0.37, y, f"{val:.2f} °C", transform=ax0.transAxes,
                 fontsize=13, fontweight='bold', color='#222', va='center',
                 family='monospace')
        y -= step

    # Histogram ------------------------------------------------------------
    ax1 = fig.add_subplot(gs[0, 1])
    _, edges, patches = ax1.hist(temps, bins=32, edgecolor='white', linewidth=0.4)
    for patch, edge_lo, edge_hi in zip(patches, edges[:-1], edges[1:]):
        patch.set_facecolor(temp_color((edge_lo + edge_hi) / 2, lo, hi))
    ax1.axvline(mean, color='#222', linestyle='--', linewidth=1.1, alpha=0.7)
    ax1.text(mean, 0.96, f'  mean {mean:.1f} °C',
             transform=ax1.get_xaxis_transform(),
             color='#222', fontsize=9, va='top')
    ax1.set_xlabel('Tile temperature (°C)')
    ax1.set_ylabel('Tile count')
    ax1.set_title('Temperature distribution across AOI')
    ax1.grid(axis='y', alpha=0.3)
    for spine in ('top', 'right'):
        ax1.spines[spine].set_visible(False)

    # Colorbar -------------------------------------------------------------
    ax2 = fig.add_subplot(gs[0, 2])
    grad = np.linspace(lo, hi, 256).reshape(-1, 1)
    ax2.imshow(grad, aspect='auto', cmap=TCM_CMAP,
               extent=[0, 1, lo, hi], origin='lower')
    ax2.set_xticks([])
    ax2.yaxis.tick_right()
    ax2.set_ylabel('°C', rotation=0, labelpad=12, fontsize=9)

    plt.show()

def _f_to_c(f):
    return (f - 32.0) * 5.0 / 9.0

def _load_cached_heatmap():
    """Return (tiles, aoi_bounds). tiles = list of (polygon, hourly_c, peak_c, peak_h, min_c)."""
    with open(HEATMAP_GEOJSON, 'r', encoding='utf-8') as f:
        gj = json.load(f)
    feats = gj.get('features', [])
    tiles = []
    minx = miny =  1e9
    maxx = maxy = -1e9
    for ft in feats:
        poly = shape(ft['geometry'])
        props = ft['properties']
        hourly_c = [_f_to_c(props[f"{h:02d}"]) for h in range(24)]
        peak_c   = max(hourly_c)
        peak_h   = hourly_c.index(peak_c)
        min_c    = min(hourly_c)
        tiles.append((poly, hourly_c, peak_c, peak_h, min_c))
        x0, y0, x1, y1 = poly.bounds
        if x0 < minx: minx = x0
        if y0 < miny: miny = y0
        if x1 > maxx: maxx = x1
        if y1 > maxy: maxy = y1
    return tiles, (minx, miny, maxx, maxy)

if CACHED:
    tiles, aoi_bounds = _load_cached_heatmap()
    peaks = [t[2] for t in tiles]
    lo, hi = min(peaks), max(peaks)
    assert TEMP_C_SANITY[0] <= lo and hi <= TEMP_C_SANITY[1], \
        f"F→C sanity check failed: peak range {lo:.1f}..{hi:.1f} outside {TEMP_C_SANITY}"
    print(f"[cached] {len(tiles)} tiles, peak temp range {lo:.1f}..{hi:.1f} °C")
    print(f"[cached] AOI bounds (lon,lat): ({aoi_bounds[0]:.4f}, {aoi_bounds[1]:.4f}) → "
          f"({aoi_bounds[2]:.4f}, {aoi_bounds[3]:.4f})")
    show_heatmap_summary(peaks, f"Cached · {HEATMAP_GEOJSON.name}")
else:
    heatmap = client.create_heatmap(
        polygon_aoi=SAN_JOSE_POLYGON, start_date=STUDY_DATE, start_time=STUDY_HOUR,
        filter_type=1, granularity=GRANULARITY_M,
    )
    map_data = heatmap['result'].get('map_data') or {}
    feats = map_data.get('features', []) if isinstance(map_data, dict) else []
    tiles = []
    minx = miny =  1e9
    maxx = maxy = -1e9
    sh = int(STUDY_HOUR.split(':')[0])
    for ft in feats:
        poly = shape(ft['geometry'])
        t = ft['properties'].get('temperature')
        # Single-hour live data: broadcast to 24-hour tuple so downstream code is uniform.
        # Diurnal columns will be degenerate in this mode — see banner below.
        hourly_c = [t] * 24
        tiles.append((poly, hourly_c, t, sh, t))
        x0, y0, x1, y1 = poly.bounds
        if x0 < minx: minx = x0
        if y0 < miny: miny = y0
        if x1 > maxx: maxx = x1
        if y1 > maxy: maxy = y1
    aoi_bounds = (minx, miny, maxx, maxy)
    peaks = [t[2] for t in tiles]
    print(f"[online] {len(tiles)} tiles at {GRANULARITY_M}m, single hour {STUDY_HOUR}.")
    print("[online] ⚠ Diurnal swing / peak_hour / aoi_percentile collapse to the snapshot "
          "hour. Use CACHED=True with a 24h GeoJSON for full-day analysis.")
    show_heatmap_summary(peaks, f"Live API · {STUDY_DATE} {STUDY_HOUR}")

def tile_for(lat, lon):
    p = Point(lon, lat)
    for t in tiles:
        if t[0].contains(p):
            return t
    return min(tiles, key=lambda t: t[0].centroid.distance(p))

### Visualize the heatmap

A spatial preview of the 24-hour **peak temperature** across every tile in the AOI. Each cell is shaded on a cool-blue → hot-red ramp by its daily maximum — hover any tile for the exact value. This is the layer the next steps join your portfolio against, so it pays to eyeball where the urban heat islands sit *before* the spatial join.

In [ ]:
# Visualize the average-temperature heatmap across the AOI using an
# **equal-interval classification** — same algorithm as the TS reference:
#     interval = (highest - least) / numberOfClasses
#     for i in 0..N-1:  class i covers [least + i*interval, least + (i+1)*interval)
#                       with color = colorRamp[i % len(colorRamp)]
# Every tile is then drawn with the color of the class its daily-average
# temperature falls into.
TCM_COLORS = [
    '#2983ba', '#5aa4b2', '#88c4aa', '#b3e0a6', '#d1ecb0', '#f0f9ba',
    '#fff0ae', '#fed38c', '#fdb56a', '#f3854e', '#e54f35', '#d7191c',
]
N_BINS = len(TCM_COLORS)

# Per-tile daily average over the 24 hourly values (hourly_c is index 1 of the tile tuple).
tile_avgs = [sum(t[1]) / len(t[1]) for t in tiles]
least, highest = min(tile_avgs), max(tile_avgs)
mean_t = sum(tile_avgs) / len(tile_avgs)
interval = (highest - least) / N_BINS if highest > least else 0.0

class_entries = [
    {'min':   least + i * interval,
     'max':   least + (i + 1) * interval,
     'color': TCM_COLORS[i % len(TCM_COLORS)]}
    for i in range(N_BINS)
]

def _class_color(t):
    """Match the TS reference: pick the class whose [min, max) contains t."""
    if t is None or interval == 0:
        return class_entries[0]['color']
    idx = int((t - least) / interval)        # floor → class index
    idx = max(0, min(idx, N_BINS - 1))       # the top edge collapses into the top class
    return class_entries[idx]['color']

heatmap_features = [{
    'type': 'Feature',
    'geometry': mapping(poly),
    'properties': {
        'avg_str': f"{avg_c:.2f} °C",
        'fillColor': _class_color(avg_c),
    },
} for (poly, _, _, _, _), avg_c in zip(tiles, tile_avgs)]

aoi_center = [(aoi_bounds[1] + aoi_bounds[3]) / 2,
              (aoi_bounds[0] + aoi_bounds[2]) / 2]
m_heat = folium.Map(location=aoi_center, zoom_start=13, tiles='cartodbpositron')

# Style each tile vivid + opaque, with a hairline stroke matched to the fill
# so adjacent cells seam cleanly into a solid raster (no basemap bleed,
# no dark gridlines from the polygon stroke).
folium.GeoJson(
    {'type': 'FeatureCollection', 'features': heatmap_features},
    style_function=lambda f: {
        'fillColor':   f['properties']['fillColor'],
        'color':       f['properties']['fillColor'],
        'weight':      0.6,
        'fillOpacity': 0.9,
        'opacity':     1.0,
    },
    tooltip=folium.GeoJsonTooltip(fields=['avg_str'], aliases=['Avg temp'], sticky=True),
).add_to(m_heat)
m_heat.fit_bounds([[aoi_bounds[1], aoi_bounds[0]], [aoi_bounds[3], aoi_bounds[2]]])

# Discrete legend — one row per equal-interval class, hottest on top.
rows = ''.join(
    f'<tr><td style="background:{c["color"]};width:18px;height:14px;'
    f'border:1px solid #888;"></td>'
    f'<td style="padding-left:8px;font-family:monospace;">'
    f'{c["min"]:.2f} – {c["max"]:.2f} °C</td></tr>'
    for c in reversed(class_entries)
)
legend_html = (
    '<div style="position:fixed;bottom:30px;left:30px;z-index:9999;'
    'background:white;padding:8px 12px;border:1px solid #888;font:12px sans-serif;">'
    '<b>Avg temperature (24 h)</b><br/>'
    f'<span style="color:#666;">equal-interval · {N_BINS} classes · '
    f'{interval:.2f} °C wide</span>'
    f'<table style="margin-top:4px;border-collapse:collapse;">{rows}</table></div>'
)
m_heat.get_root().html.add_child(folium.Element(legend_html))

print(f"Equal-interval classification on daily average: {N_BINS} classes of "
      f"width {interval:.2f} °C over {least:.2f}..{highest:.2f} °C "
      f"(AOI mean {mean_t:.2f} °C, {len(tiles):,} tiles)")
m_heat

---
## Step 3 — Diurnal temperature attach

### What you are doing
For each property, find the tile that contains it, copy off the **peak temperature**, **peak hour**, **min temperature**, **diurnal swing**, and the **AOI percentile** (this property's peak relative to all 16,507 city tiles). Now every property has an analysis-ready row.

### Why this matters
This is the moment your portfolio table becomes a *risk* table. Every downstream question — ranking, scoring, business translation — is a `groupby` or sort on this DataFrame.

In [4]:
def _percentile_rank(value, sorted_values):
    lo, hi = 0, len(sorted_values)
    while lo < hi:
        mid = (lo + hi) // 2
        if sorted_values[mid] <= value: lo = mid + 1
        else: hi = mid
    return round(100.0 * lo / max(1, len(sorted_values)), 1)

aoi_peak_sorted = sorted(t[2] for t in tiles)

records = []
for _, r in portfolio.iterrows():
    poly, hourly_c, peak_c, peak_h, min_c = tile_for(r.latitude, r.longitude)
    records.append({
        'peak_temp_c'   : round(peak_c, 1),
        'peak_hour'     : peak_h,
        'min_temp_c'    : round(min_c, 1),
        'diurnal_swing_c': round(peak_c - min_c, 1),
        'aoi_percentile': _percentile_rank(peak_c, aoi_peak_sorted),
    })
portfolio = pd.concat([portfolio.reset_index(drop=True), pd.DataFrame(records)], axis=1)
portfolio = portfolio.sort_values('peak_temp_c', ascending=False).reset_index(drop=True)
portfolio.insert(0, 'temp_rank', portfolio.index + 1)

cols_t1 = ['temp_rank', 'property_id', 'name', 'type', 'sqft',
           'peak_temp_c', 'peak_hour', 'min_temp_c', 'diurnal_swing_c', 'aoi_percentile']
portfolio[cols_t1]

,temp_rank,property_id,name,type,sqft,peak_temp_c,peak_hour,min_temp_c,diurnal_swing_c,aoi_percentile
0,1,P01,Adobe Campus Tower,Office,620000,40.7,15,18.7,22.1,99.5
1,2,P08,Southside Industrial,Industrial,180000,40.5,15,19.3,21.2,76.6
2,3,P09,The Fairmont Residences,Residential,140000,40.1,15,19.3,20.8,58.1
3,4,P07,Paseo Office Center,Office,320000,40.1,15,19.3,20.8,58.1
4,5,P02,Downtown Commons Residences,Residential,340000,39.8,16,19.3,20.5,42.3
5,6,P03,City View Office Plaza,Office,450000,39.8,16,19.3,20.5,42.3
6,7,P06,Japantown Lofts,Residential,95000,39.8,16,19.3,20.5,36.2
7,8,P05,San Pedro Square Mixed-Use,Mixed-Use,220000,39.8,16,19.3,20.5,37.3
8,9,P04,Santa Clara Street Retail,Retail,45000,39.8,16,19.3,20.5,42.3
9,10,P10,Tech Corridor Offices,Office,280000,39.8,16,19.2,20.6,43.5


---
## Step 4 — Portfolio overview map (M1)

### What you are doing
Two layered views.

- **M1a** drops the full AOI heatmap (the equal-interval classes from Step 2, on daily-average °C) under the portfolio. Marker size scales with peak temperature; color encodes asset type.
- **M1b** isolates the spatial join: only the heatmap tiles that strictly contain a portfolio asset are drawn, with the same markers on top. Properties whose coordinates do not fall inside any tile (i.e., outside the AOI) are flagged.

### Why this matters
M1a is the slide-1 visual for the client meeting — before any score is computed, the agent can already see which assets sit in hot zones. M1b makes the join itself legible: every property is bound to one specific tile temperature, and that single number is the foundation for everything downstream.

In [ ]:
def _legend_html(palette):
    rows = ''.join(
        f'<tr><td style="background:{c};width:18px;"></td>'
        f'<td style="padding-left:6px;">{name}</td></tr>'
        for name, c in palette.items()
    )
    return (
        '<div style="position:fixed;bottom:30px;left:30px;z-index:9999;'
        'background:white;padding:8px 12px;border:1px solid #888;'
        'font:12px sans-serif;">'
        '<b>Asset type</b>'
        f'<table>{rows}</table>'
        '</div>'
    )

center = [portfolio['latitude'].mean(), portfolio['longitude'].mean()]
min_peak = portfolio['peak_temp_c'].min()

def _add_property_marker(m, p):
    folium.CircleMarker(
        location=[p.latitude, p.longitude],
        radius=4 + (p.peak_temp_c - min_peak) * 1.2,
        color='#000', weight=1.2,
        fill=True, fill_color=ASSET_TYPE_PALETTE.get(p['type'], '#888'),
        fill_opacity=0.95,
        popup=(f"<b>{p['name']}</b><br/>"
               f"{p['type']}, {p['sqft']:,} sqft, ${p['market_value_musd']}M<br/>"
               f"peak: {p.peak_temp_c:.1f}°C @ {int(p.peak_hour):02d}:00<br/>"
               f"AOI percentile: {p.aoi_percentile}"),
    ).add_to(m)

# ── M1a — full AOI heatmap with every portfolio point on top ──────────────────
m1a = folium.Map(location=center, zoom_start=13, tiles='cartodbpositron')
folium.GeoJson(
    {'type': 'FeatureCollection', 'features': heatmap_features},
    style_function=lambda f: {
        'fillColor':   f['properties']['fillColor'],
        'color':       f['properties']['fillColor'],
        'weight':      0.6,
        'fillOpacity': 0.75,
        'opacity':     1.0,
    },
    tooltip=folium.GeoJsonTooltip(fields=['avg_str'], aliases=['Avg temp'], sticky=True),
).add_to(m1a)
for _, p in portfolio.iterrows():
    _add_property_marker(m1a, p)
m1a.get_root().html.add_child(folium.Element(_legend_html(ASSET_TYPE_PALETTE)))
display(m1a)

# ── M1b — only the tiles a portfolio asset actually sits on ───────────────────
# Spatial-join footprint: each property maps to exactly one tile (or none, if
# its coordinates fall outside the AOI). We render only those tiles, colored
# by the same equal-interval classes, with the markers stacked on top.
def _strict_tile_for(lat, lon):
    """Return the tile that strictly contains the point, or None."""
    pt = Point(lon, lat)
    for t in tiles:
        if t[0].contains(pt):
            return t
    return None

joined_tiles, seen = [], set()
matched_props, unmatched_props = [], []
for _, p in portfolio.iterrows():
    t = _strict_tile_for(p.latitude, p.longitude)
    if t is None:
        unmatched_props.append(p)
        continue
    matched_props.append(p)
    poly, hourly_c, _, _, _ = t
    key = (round(poly.centroid.x, 6), round(poly.centroid.y, 6))
    if key in seen:
        continue
    seen.add(key)
    avg_c = sum(hourly_c) / len(hourly_c)
    joined_tiles.append({
        'type': 'Feature',
        'geometry': mapping(poly),
        'properties': {
            'avg_str':   f"{avg_c:.2f} °C",
            'fillColor': _class_color(avg_c),
        },
    })

print(f"M1b spatial join — {len(matched_props)} / {len(portfolio)} properties "
      f"sit on a heatmap tile ({len(joined_tiles)} unique tiles).")
if unmatched_props:
    names = ', '.join(f"{p['property_id']} ({p['name']})" for p in unmatched_props)
    print(f"  ⚠ Outside AOI / no containing tile: {names}")

m1b = folium.Map(location=center, zoom_start=14, tiles='cartodbpositron')
folium.GeoJson(
    {'type': 'FeatureCollection', 'features': joined_tiles},
    style_function=lambda f: {
        'fillColor':   f['properties']['fillColor'],
        'color':       '#000',
        'weight':      2.0,
        'fillOpacity': 0.9,
        'opacity':     1.0,
    },
    tooltip=folium.GeoJsonTooltip(fields=['avg_str'], aliases=['Tile avg'], sticky=True),
).add_to(m1b)
for p in matched_props:
    _add_property_marker(m1b, p)
m1b.fit_bounds([[portfolio['latitude'].min() - 0.005,
                 portfolio['longitude'].min() - 0.005],
                [portfolio['latitude'].max() + 0.005,
                 portfolio['longitude'].max() + 0.005]])
m1b.get_root().html.add_child(folium.Element(_legend_html(ASSET_TYPE_PALETTE)))
m1b

---
## Step 5 — Above-median hot exposures (M2)

### What you are doing
Two views, in order:

- **M2a** — AOI-wide heatmap classified on each tile's **24-h peak** (not the daily average from Step 2), with the full portfolio dropped on top. Every property is placed in the city's peak-temperature context.
- **M2b** — Drill-down to just the properties whose peak temperature is at or above the portfolio median, rendering only the heatmap tiles those assets sit on (M1b-style spatial join).

### Why this matters
M2a frames the conversation against the city — "your portfolio sits in *this* slice of San Jose at peak hour." M2b cuts the noise: the agent doesn't need ten talking points, they need a few. Together they are the slide-2 visual that says: "these are the assets we have to discuss, and here's how they compare to the city peak."

In [ ]:
# M2a — AOI-wide PEAK-temperature heatmap with the full portfolio on top.
# Same equal-interval scheme as Step 2, but classes are computed on each
# tile's 24-h max (not the daily average), so the legend tells you which
# slice of the city hits which peak.
tile_peaks    = [t[2] for t in tiles]
peak_least    = min(tile_peaks)
peak_highest  = max(tile_peaks)
peak_interval = (peak_highest - peak_least) / N_BINS if peak_highest > peak_least else 0.0
peak_classes  = [
    {'min':   peak_least + i * peak_interval,
     'max':   peak_least + (i + 1) * peak_interval,
     'color': TCM_COLORS[i % len(TCM_COLORS)]}
    for i in range(N_BINS)
]

def _peak_class_color(t):
    if t is None or peak_interval == 0:
        return peak_classes[0]['color']
    idx = int((t - peak_least) / peak_interval)
    return peak_classes[max(0, min(idx, N_BINS - 1))]['color']

peak_features = [{
    'type': 'Feature',
    'geometry': mapping(poly),
    'properties': {
        'peak_str':  f"{peak_c:.2f} °C peak",
        'fillColor': _peak_class_color(peak_c),
    },
} for poly, _, peak_c, _, _ in tiles]

m2a = folium.Map(location=center, tiles='cartodbpositron')
folium.GeoJson(
    {'type': 'FeatureCollection', 'features': peak_features},
    style_function=lambda f: {
        'fillColor':   f['properties']['fillColor'],
        'color':       f['properties']['fillColor'],
        'weight':      0.6,
        'fillOpacity': 0.85,
        'opacity':     1.0,
    },
    tooltip=folium.GeoJsonTooltip(fields=['peak_str'], aliases=['Tile peak'], sticky=True),
).add_to(m2a)
for _, p in portfolio.iterrows():
    folium.CircleMarker(
        location=[p.latitude, p.longitude],
        radius=4 + (p.peak_temp_c - min_peak) * 1.2,
        color='black', weight=1.2,
        fill=True, fill_color=ASSET_TYPE_PALETTE.get(p['type'], '#888'), fill_opacity=0.95,
        tooltip=f"#{int(p.temp_rank)} {p['property_id']} — {p.peak_temp_c:.1f}°C",
        popup=(f"<b>#{int(p.temp_rank)} {p['name']}</b><br/>"
               f"{p['type']}, {p['sqft']:,} sqft, ${p['market_value_musd']}M<br/>"
               f"peak: {p.peak_temp_c:.1f}°C @ {int(p.peak_hour):02d}:00<br/>"
               f"AOI percentile: {p.aoi_percentile}"),
    ).add_to(m2a)
m2a.fit_bounds([[aoi_bounds[1], aoi_bounds[0]], [aoi_bounds[3], aoi_bounds[2]]])

# Combined legend — peak-temp classes + asset-type swatches.
temp_rows_p = ''.join(
    f'<tr><td style="background:{c["color"]};width:18px;height:14px;'
    f'border:1px solid #888;"></td>'
    f'<td style="padding-left:8px;font-family:monospace;">'
    f'{c["min"]:.2f} – {c["max"]:.2f} °C</td></tr>'
    for c in reversed(peak_classes)
)
type_rows_p = ''.join(
    f'<tr><td style="background:{c};width:14px;height:14px;'
    f'border:1px solid #000;border-radius:50%;"></td>'
    f'<td style="padding-left:8px;">{name}</td></tr>'
    for name, c in ASSET_TYPE_PALETTE.items()
)
legend_html_p = (
    '<div style="position:fixed;bottom:30px;left:30px;z-index:9999;'
    'background:white;padding:8px 12px;border:1px solid #888;font:12px sans-serif;">'
    '<b>Tile peak temp (24 h)</b><br/>'
    f'<span style="color:#666;">equal-interval · {N_BINS} classes · '
    f'{peak_interval:.2f} °C wide · {peak_least:.2f}..{peak_highest:.2f} °C</span>'
    f'<table style="margin-top:4px;border-collapse:collapse;">{temp_rows_p}</table>'
    '<b style="display:block;margin-top:6px;">Asset type</b>'
    f'<table style="margin-top:4px;border-collapse:collapse;">{type_rows_p}</table>'
    '</div>'
)
m2a.get_root().html.add_child(folium.Element(legend_html_p))

print(f"M2a — AOI peak heatmap: {len(tiles):,} tiles, "
      f"peak range {peak_least:.2f}..{peak_highest:.2f} °C, "
      f"{N_BINS} classes of width {peak_interval:.2f} °C")
m2a

In [ ]:
median_peak = portfolio['peak_temp_c'].median()
hot = portfolio[portfolio['peak_temp_c'] >= median_peak].copy()

# Spatial join restricted to the hot subset: each above-median property
# pins to exactly one heatmap tile (the one its lat/lon lands inside).
# We render ONLY those tiles — same pattern as M1b — so the overlay
# answers "what tile does each hot asset sit on?" and nothing else.
def _strict_tile_for(lat, lon):
    pt = Point(lon, lat)
    for t in tiles:
        if t[0].contains(pt):
            return t
    return None

joined, seen = [], set()
matched_props, unmatched_props = [], []
for _, p in hot.iterrows():
    t = _strict_tile_for(p.latitude, p.longitude)
    if t is None:
        unmatched_props.append(p)
        continue
    matched_props.append(p)
    poly, _, peak_c, peak_h, _ = t
    key = (round(poly.centroid.x, 6), round(poly.centroid.y, 6))
    if key in seen:
        continue
    seen.add(key)
    joined.append({'poly': poly, 'peak_c': peak_c, 'peak_h': peak_h})

# Equal-interval classes recomputed on just the joined tiles so the
# legend describes the overlay (and not the full AOI).
joined_peaks = [j['peak_c'] for j in joined] or [median_peak]
least_h    = min(joined_peaks)
highest_h  = max(joined_peaks)
interval_h = (highest_h - least_h) / N_BINS if highest_h > least_h else 0.0
hot_classes = [
    {'min':   least_h + i * interval_h,
     'max':   least_h + (i + 1) * interval_h,
     'color': TCM_COLORS[i % len(TCM_COLORS)]}
    for i in range(N_BINS)
]

def _hot_class_color(t):
    if t is None or interval_h == 0:
        return hot_classes[-1]['color']
    idx = int((t - least_h) / interval_h)
    return hot_classes[max(0, min(idx, N_BINS - 1))]['color']

joined_features = [{
    'type': 'Feature',
    'geometry': mapping(j['poly']),
    'properties': {
        'peak_str':  f"{j['peak_c']:.2f} °C peak @ {j['peak_h']:02d}:00",
        'fillColor': _hot_class_color(j['peak_c']),
    },
} for j in joined]

m2 = folium.Map(location=center, tiles='cartodbpositron')
folium.GeoJson(
    {'type': 'FeatureCollection', 'features': joined_features},
    style_function=lambda f: {
        'fillColor':   f['properties']['fillColor'],
        'color':       '#000',
        'weight':      1.5,
        'fillOpacity': 0.9,
        'opacity':     1.0,
    },
    tooltip=folium.GeoJsonTooltip(fields=['peak_str'], aliases=['Tile peak'], sticky=True),
).add_to(m2)
for p in matched_props:
    folium.CircleMarker(
        location=[p.latitude, p.longitude],
        radius=8, color='black', weight=1.2,
        fill=True, fill_color=ASSET_TYPE_PALETTE.get(p['type'], '#888'), fill_opacity=0.95,
        tooltip=f"#{int(p.temp_rank)} {p['property_id']} — {p.peak_temp_c:.1f}°C",
        popup=(f"<b>#{int(p.temp_rank)} {p['name']}</b><br/>"
               f"{p['type']}<br/>"
               f"peak: {p.peak_temp_c:.1f}°C @ {int(p.peak_hour):02d}:00<br/>"
               f"AOI percentile: {p.aoi_percentile}"),
    ).add_to(m2)

# Frame on the markers (with a small pad for context).
hot_lats = [p.latitude  for p in matched_props] or [hot['latitude'].mean()]
hot_lons = [p.longitude for p in matched_props] or [hot['longitude'].mean()]
lat_lo, lat_hi = min(hot_lats), max(hot_lats)
lon_lo, lon_hi = min(hot_lons), max(hot_lons)
pad_lat = max(0.004, (lat_hi - lat_lo) * 0.20)
pad_lon = max(0.004, (lon_hi - lon_lo) * 0.20)
m2.fit_bounds([[lat_lo - pad_lat, lon_lo - pad_lon],
               [lat_hi + pad_lat, lon_hi + pad_lon]])

# Combined legend — temperature classes (tile fill) + asset-type swatches (markers).
temp_rows = ''.join(
    f'<tr><td style="background:{c["color"]};width:18px;height:14px;'
    f'border:1px solid #888;"></td>'
    f'<td style="padding-left:8px;font-family:monospace;">'
    f'{c["min"]:.2f} – {c["max"]:.2f} °C</td></tr>'
    for c in reversed(hot_classes)
)
type_rows = ''.join(
    f'<tr><td style="background:{c};width:14px;height:14px;'
    f'border:1px solid #000;border-radius:50%;"></td>'
    f'<td style="padding-left:8px;">{name}</td></tr>'
    for name, c in ASSET_TYPE_PALETTE.items()
)
legend_html = (
    '<div style="position:fixed;bottom:30px;left:30px;z-index:9999;'
    'background:white;padding:8px 12px;border:1px solid #888;font:12px sans-serif;">'
    '<b>Tile peak temp (24 h)</b><br/>'
    f'<span style="color:#666;">portfolio-tile join · {N_BINS} equal-interval classes · '
    f'{interval_h:.2f} °C wide</span>'
    f'<table style="margin-top:4px;border-collapse:collapse;">{temp_rows}</table>'
    '<b style="display:block;margin-top:6px;">Asset type</b>'
    f'<table style="margin-top:4px;border-collapse:collapse;">{type_rows}</table>'
    '</div>'
)
m2.get_root().html.add_child(folium.Element(legend_html))

print(f"{len(hot)} above-median exposures (median peak {median_peak:.1f}°C); "
      f"{len(joined)} unique tiles after spatial join "
      f"(peak {least_h:.1f}–{highest_h:.1f}°C).")
if unmatched_props:
    names = ', '.join(f"{p['property_id']} ({p['name']})" for p in unmatched_props)
    print(f"  ⚠ Outside AOI / no containing tile: {names}")
display(m2)
hot[['temp_rank','property_id','name','type','latitude','longitude',
     'peak_temp_c','peak_hour','aoi_percentile']]

---
## Step 6 — Surface diagnosis (satellite segmentation)

### What you are doing
For the top-N hottest properties, characterize the surface mix in their immediate surroundings: building, road, sidewalk, tree, grass. We bucket those into `impervious_pct` and `vegetation_pct` for scoring, and chart the full breakdown.

### Why this matters
Temperature alone doesn't tell you what to spend money on. A hot property surrounded by impervious rooftops is a cool-roof candidate. A hot property with minimal vegetation is a planting candidate. The surface mix points to the lever — without us prescribing which one to pull.

In [ ]:
IMPERV_KEYS = {'road', 'roads', 'pavement', 'building', 'buildings',
               'rooftop', 'rooftops', 'sidewalk', 'earth', 'bare', 'ground'}
VEGGIE_KEYS = {'vegetation', 'tree', 'trees', 'grass', 'greenery', 'park'}

def _bucket(segments, keys):
    total = 0.0
    for cls, pct in (segments or {}).items():
        if any(k in cls.lower() for k in keys):
            try: total += float(pct)
            except (TypeError, ValueError): pass
    return round(total, 1)

def _first_b64(value):
    if isinstance(value, list):
        return value[0] if value else None
    return value

top_n = portfolio.head(TOP_N_TO_ENRICH).copy()
seg_data = {}   # property_id -> segments dict
sat_imgs = {}   # property_id -> {'orig': b64, 'seg': b64}

if CACHED:
    missing = []
    for _, r in top_n.iterrows():
        pid = r.property_id
        cached_path = SATELLITE_JSON.with_name(
            SATELLITE_JSON.stem + f'_{pid.lower()}' + SATELLITE_JSON.suffix
        )
        if not cached_path.exists():
            missing.append(pid)
            continue
        with open(cached_path, 'r', encoding='utf-8') as f:
            sat_doc = json.load(f)
        seg_block = sat_doc.get('segmentation', {}) or {}
        seg_data[pid] = seg_block.get('segments', {}) or {}
        sat_imgs[pid] = {
            'orig': _first_b64(sat_doc.get('orignal_image')),
            'seg':  seg_block.get('image_content'),
        }
    if missing:
        print(f"⚠ Cached satellite missing for {missing}; set CACHED=False to fetch live.")
else:
    for _, r in top_n.iterrows():
        print(f"  satellite: #{int(r.temp_rank)} {r.property_id}")
        sat = client.satellite_segmentation(
            latitude=r.latitude, longitude=r.longitude,
            start_date=STUDY_DATE, start_time=STUDY_HOUR,
            filter_type=1, granularity=GRANULARITY_M, verbose=False,
        )
        res = sat.get('result', {}) or {}
        seg_block = res.get('segmentation', {}) or {}
        seg_data[r.property_id] = seg_block.get('segments', {}) or {}
        sat_imgs[r.property_id] = {
            'orig': _first_b64(res.get('orignal_image')),
            'seg':  seg_block.get('image_content'),
        }

top_n['impervious_pct'] = top_n['property_id'].map(
    lambda pid: _bucket(seg_data.get(pid), IMPERV_KEYS) if pid in seg_data else None)
top_n['vegetation_pct'] = top_n['property_id'].map(
    lambda pid: _bucket(seg_data.get(pid), VEGGIE_KEYS) if pid in seg_data else None)

enriched_rows = [(pid, segs) for pid, segs in seg_data.items() if segs]
if enriched_rows:
    classes = sorted({c for _, s in enriched_rows for c in s.keys()})
    fig, ax = plt.subplots(figsize=(8, max(2.5, 0.8 * len(enriched_rows))))
    bottoms = [0.0] * len(enriched_rows)
    pids    = [pid for pid, _ in enriched_rows]
    for c in classes:
        vals = [float(segs.get(c, 0.0)) for _, segs in enriched_rows]
        ax.barh(pids, vals, left=bottoms, label=c)
        bottoms = [b + v for b, v in zip(bottoms, vals)]
    ax.set_xlabel('% of surrounding scene')
    ax.set_title('C2 — Satellite surface composition (top exposures)')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout(); plt.show()

def _sat_img_tag(b64, label):
    if not b64:
        return (f'<div style="padding:1em;border:1px dashed #888;'
                f'width:300px;height:300px;display:flex;align-items:center;'
                f'justify-content:center;font:11px sans-serif">{label}: missing</div>')
    return (f'<figure style="margin:0">'
            f'<img src="data:image/png;base64,{b64}" '
            f'style="width:300px;border:1px solid #888;"/>'
            f'<figcaption style="text-align:center;font:11px sans-serif">{label}</figcaption>'
            f'</figure>')

img_blocks = []
for _, r in top_n.iterrows():
    pid = r.property_id
    imgs = sat_imgs.get(pid)
    if not imgs:
        continue
    header = (f'<div style="font:12px sans-serif;margin:8px 0 4px 0">'
              f'<b>#{int(r.temp_rank)} {pid}</b> — {r["name"]} '
              f'({r.peak_temp_c:.1f}°C peak)</div>')
    row = ('<div style="display:flex;gap:12px;flex-wrap:wrap;margin-bottom:12px">'
           + _sat_img_tag(imgs.get('orig'), 'original')
           + _sat_img_tag(imgs.get('seg'),  'segmented')
           + '</div>')
    img_blocks.append(header + row)
if img_blocks:
    display(HTML(''.join(img_blocks)))

top_n[['temp_rank','property_id','name','peak_temp_c','impervious_pct','vegetation_pct']]

---
## Step 7 — Street-view ground truth on #1

### What you are doing
Pulling the front-facing street view at the highest-ranked enriched property and rendering the original alongside the segmented version. Tabulating the front-view composition (sky, building, tree, road, sidewalk, car, grass).

### Why this matters
Satellite shows the roof; the agent's client walks past the curb. Sky-fraction tells you canyon openness; car-fraction tells you traffic load; tree-fraction at the curb is what a passer-by experiences. This step grounds the satellite story in something a non-engineer recognizes.

In [ ]:
def _sv_payload(front):
    front = front or {}
    return {
        'orig'      : front.get('original_image'),
        'seg'       : front.get('segmented_image'),
        'segs'      : front.get('segments', {}) or {},
        'image_date': front.get('image_date', 'n/a'),
    }

sv_data = {}  # property_id -> {'orig','seg','segs','image_date'}

if CACHED:
    sv_missing = []
    for _, r in top_n.iterrows():
        pid = r.property_id
        path = STREETVIEW_JSON.with_name(
            STREETVIEW_JSON.stem + f'_{pid.lower()}' + STREETVIEW_JSON.suffix
        )
        if not path.exists():
            sv_missing.append(pid)
            continue
        with open(path, 'r', encoding='utf-8') as f:
            sv_doc = json.load(f)
        sv_data[pid] = _sv_payload(sv_doc.get('front'))
    if sv_missing:
        print(f"⚠ Cached street-view missing for {sv_missing}; set CACHED=False to fetch live.")
else:
    for _, r in top_n.iterrows():
        print(f"  street-view: #{int(r.temp_rank)} {r.property_id}")
        sv_resp = client.street_view_segmentation(
            latitude=r.latitude, longitude=r.longitude,
            start_date=STUDY_DATE, start_time=STUDY_HOUR,
            filter_type=1, granularity=GRANULARITY_M, verbose=False,
        )
        sv_data[r.property_id] = _sv_payload((sv_resp.get('result') or {}).get('front'))

# "Property #1" = highest-rank property that actually has street-view imagery
enriched_subset = top_n[top_n['property_id'].isin(sv_data.keys())] if sv_data else top_n
property_one = enriched_subset.iloc[0] if len(enriched_subset) else top_n.iloc[0]

if sv_data:
    print("Street-view loaded for: " +
          ", ".join(f"{pid} ({d['image_date']})" for pid, d in sv_data.items()))

def _img_tag(b64, label):
    if not b64:
        return f'<div style="padding:1em;border:1px dashed #888;width:350px">{label}: missing</div>'
    return (f'<figure style="margin:0">'
            f'<img src="data:image/jpeg;base64,{b64}" '
            f'style="width:350px;border:1px solid #888;"/>'
            f'<figcaption style="text-align:center;font:11px sans-serif">{label}</figcaption>'
            f'</figure>')

img_blocks = []
for _, r in top_n.iterrows():
    pid = r.property_id
    d = sv_data.get(pid)
    if not d:
        continue
    header = (f'<div style="font:12px sans-serif;margin:8px 0 4px 0">'
              f'<b>#{int(r.temp_rank)} {pid}</b> — {r["name"]} '
              f'(imagery {d["image_date"]})</div>')
    row = ('<div style="display:flex;gap:12px;flex-wrap:wrap;margin-bottom:12px">'
           + _img_tag(d['orig'], 'original') + _img_tag(d['seg'], 'segmented')
           + '</div>')
    img_blocks.append(header + row)
if img_blocks:
    display(HTML(''.join(img_blocks)))

# C2b — stacked-bar street-view scene composition across top-N (parallel to C2 in Step 6)
sv_rows = [(r.property_id, sv_data[r.property_id]['segs'])
           for _, r in top_n.iterrows()
           if r.property_id in sv_data and sv_data[r.property_id].get('segs')]
if sv_rows:
    classes = sorted({c for _, s in sv_rows for c in s.keys()})
    fig, ax = plt.subplots(figsize=(8, max(2.5, 0.8 * len(sv_rows))))
    bottoms = [0.0] * len(sv_rows)
    pids    = [pid for pid, _ in sv_rows]
    for c in classes:
        vals = [float(segs.get(c, 0.0)) for _, segs in sv_rows]
        ax.barh(pids, vals, left=bottoms, label=c)
        bottoms = [b + v for b, v in zip(bottoms, vals)]
    ax.set_xlabel('% of street-view scene')
    ax.set_title('C2b — Street-view scene composition (front view)')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout(); plt.show()

# Per-property segments tables (front view)
for _, r in top_n.iterrows():
    pid = r.property_id
    d = sv_data.get(pid)
    if not d or not d.get('segs'):
        continue
    display(HTML(
        f'<div style="font:12px sans-serif;margin:10px 0 2px 0">'
        f'<b>#{int(r.temp_rank)} {pid}</b> — {r["name"]} (front view)</div>'
    ))
    display(pd.DataFrame(sorted(d['segs'].items(), key=lambda kv: -kv[1]),
                         columns=['class', 'pct']))

---
## Step 8 — Diurnal driver profile (env-params)

### What you are doing
Pulling the full-day environmental parameters for the top-N exposures: heat index, apparent temperature, relative humidity, solar irradiance. Computing per property: peak heat index, peak apparent temperature, hours-above-SLA in business hours (09:00–18:00), peak solar irradiance.

### Why this matters
Tenant-comfort SLAs are written in terms of heat index, not dry-bulb temperature. A building at 34 °C ambient with low humidity is comfortable; one at 31 °C with 80 % humidity is not. Heat index is the number complaints cluster around — and it's what the leasing team will quote back to you.

In [ ]:
def _slice_business_hours(values):
    return [values[h] for h in range(9, 19)]  # 09:00..18:00 inclusive

def _env_series(loc):
    p = (loc or {}).get('parameters', {}) or {}
    return {
        'heat_index_celsius'           : list(p.get('heat_index_celsius') or []),
        'apparent_temperature_celsius' : list(p.get('apparent_temperature_celsius') or []),
        'relative_humidity_percent'    : list(p.get('relative_humidity_percent') or []),
        # Solar may be a 24-hour series OR a daytime-aggregate dict (clear_sky.ghi/dni/dhi).
        'solar_irradiance'             : (loc or {}).get('solar_irradiance'),
    }

env_data = {}  # property_id -> dict of full-day series

if CACHED:
    env_missing = []
    for _, r in top_n.iterrows():
        pid = r.property_id
        path = ENV_PARAMS_JSON.with_name(
            ENV_PARAMS_JSON.stem + f'_{pid.lower()}' + ENV_PARAMS_JSON.suffix
        )
        if not path.exists():
            env_missing.append(pid)
            continue
        with open(path, 'r', encoding='utf-8') as f:
            env_doc = json.load(f)
        locs = env_doc.get('locations') or []
        if not locs:
            continue
        env_data[pid] = _env_series(locs[0])
    if env_missing:
        print(f"⚠ Cached env-params missing for {env_missing}; set CACHED=False to fetch live.")
else:
    for _, r in top_n.iterrows():
        print(f"  env-params: #{int(r.temp_rank)} {r.property_id}")
        env = client.environmental_parameters(
            latitude=r.latitude, longitude=r.longitude,
            temperature=float(r.peak_temp_c),
            start_date=STUDY_DATE, start_time='09:00', end_time='18:00',
            filter_type=2, verbose=False,
        )
        loc = env['result']['locations'][0]
        env_data[r.property_id] = _env_series(loc)

def _peak_solar(sol_raw):
    if isinstance(sol_raw, list) and sol_raw:
        return max(sol_raw)
    if isinstance(sol_raw, dict):
        return (sol_raw.get('clear_sky') or {}).get('ghi')
    return None

def _peak_metrics(s):
    hi   = s.get('heat_index_celsius') or []
    appt = s.get('apparent_temperature_celsius') or []
    business = _slice_business_hours(hi) if len(hi) >= 19 else hi
    return {
        'peak_heat_index_c'    : max(hi) if hi else None,
        'peak_apparent_temp_c' : max(appt) if appt else None,
        'hours_above_sla'      : sum(1 for v in business if v is not None and v > SLA_HI_C),
        'peak_solar_irradiance': _peak_solar(s.get('solar_irradiance')),
    }

metrics = {pid: _peak_metrics(s) for pid, s in env_data.items()}
for col in ('peak_heat_index_c', 'peak_apparent_temp_c', 'hours_above_sla', 'peak_solar_irradiance'):
    top_n[col] = top_n['property_id'].map(lambda p, _c=col: (metrics.get(p) or {}).get(_c))

# C3 — diurnal driver profile, one figure per property in temp_rank order
def _draw_diurnal(ax_top, ax_bot, s, title):
    n = len(s.get('heat_index_celsius') or [])
    hours = list(range(n))
    if s.get('heat_index_celsius'):
        ax_top.plot(hours, s['heat_index_celsius'], label='Heat index (°C)', color='#d62728')
    if s.get('apparent_temperature_celsius'):
        ax_top.plot(hours, s['apparent_temperature_celsius'], label='Apparent temp (°C)', color='#ff7f0e')
    ax_top.axhline(SLA_HI_C, color='#888', linestyle='--', linewidth=1, label=f'SLA ({SLA_HI_C}°C)')
    ax_top.set_ylabel('°C')
    ax_top.set_xlabel('Hour of day')
    ax_top.legend(loc='upper left', fontsize=9)
    if s.get('relative_humidity_percent'):
        ax_rh = ax_top.twinx()
        ax_rh.plot(hours, s['relative_humidity_percent'], label='RH (%)', color='#1f77b4', alpha=0.6)
        ax_rh.set_ylabel('Relative humidity (%)', color='#1f77b4')
        ax_rh.legend(loc='upper right', fontsize=9)

    sol_raw = s.get('solar_irradiance')
    if isinstance(sol_raw, list) and sol_raw:
        ax_bot.plot(range(len(sol_raw)), sol_raw, color='#bcbd22')
        ax_bot.set_xlabel('Hour of day')
        ax_bot.set_ylabel('Solar irradiance (W/m²)')
    elif isinstance(sol_raw, dict) and sol_raw.get('clear_sky'):
        cs = sol_raw['clear_sky']
        ax_bot.bar(['GHI', 'DNI', 'DHI'],
                   [cs.get('ghi', 0), cs.get('dni', 0), cs.get('dhi', 0)],
                   color=['#bcbd22', '#e377c2', '#7f7f7f'])
        ax_bot.set_ylabel('W/m² (clear-sky daytime avg)')
        ax_bot.set_title('Clear-sky solar components')
    else:
        ax_bot.text(0.5, 0.5, 'No solar irradiance data', transform=ax_bot.transAxes,
                    ha='center', va='center', color='#888')
        ax_bot.set_xticks([]); ax_bot.set_yticks([])

    ax_top.set_title(title)

for _, r in top_n.iterrows():
    pid = r.property_id
    if pid not in env_data:
        continue
    fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(9, 6))
    _draw_diurnal(ax_top, ax_bot, env_data[pid],
                  f"C3 — Diurnal drivers — #{int(r.temp_rank)} {pid} ({r['name']})")
    plt.tight_layout(); plt.show()

top_n[['temp_rank','property_id','name','peak_temp_c',
       'peak_heat_index_c','peak_apparent_temp_c','hours_above_sla']]

---
## Step 9 — Action brief

### What you are doing
For each of the top-N hottest properties, write a short plain-English brief that says **what's happening, why, and what to do next** — and stop there. Then one final map showing the same recommendation pinned to each property.

### Why this matters
Steps 1–8 produced everything an analyst needs: ranked exposures, surface composition, ground-truth imagery, diurnal driver curves. The real-estate operator on the receiving end does not need another score, table, or quadrant chart — they need to know **which building to fix first and what to do about it.** This step delivers that and nothing else.

Every recommendation cites a public intervention program (EPA Heat Island Reduction, USDA i-Tree, ASHRAE 90.1, ASHRAE 55, OSHA Heat Illness Prevention). The triggering condition is a measurement from Step 6 or Step 8 — never a constructed score.

In [ ]:
# Step 9 — Action brief: one plain-English paragraph per top-N property +
# one ranked map. Nothing else. Every recommendation cites a public program.

NOAA_EXTREME_CAUTION_C = 32.0   # NOAA NWS Heat Index 'Extreme Caution' threshold

def _pick_action(p):
    """Single most-relevant published intervention for this property's measurements."""
    imp = p.impervious_pct    if pd.notna(p.impervious_pct)    else None
    veg = p.vegetation_pct    if pd.notna(p.vegetation_pct)    else None
    hi  = p.peak_heat_index_c if pd.notna(p.peak_heat_index_c) else None

    if imp is not None and imp >= 70:
        return ("Cool-roof retrofit",
                "high-albedo roofing on a building surrounded by paved or built surface "
                "is the single highest-leverage move at this site",
                "EPA Heat Island Reduction · ASHRAE 90.1-2022 roof solar-reflectance")
    if veg is not None and veg < 20:
        return ("Shade-tree planting",
                "canopy expansion in a low-vegetation site lowers ambient temperature "
                "and shades the building envelope",
                "USDA Forest Service i-Tree · EPA Heat Island Reduction (Trees & Vegetation)")
    if hi is not None and hi >= NOAA_EXTREME_CAUTION_C:
        return ("HVAC capacity & indoor-comfort review",
                "peak heat index breaches NOAA's Extreme Caution threshold — verify "
                "the building still meets ASHRAE 55 indoor comfort during business hours",
                "ASHRAE 55-2020 · ASHRAE 90.1 energy compliance")
    return ("Annual monitoring only",
            "no published-threshold flags raised at this site this run",
            "—")

def _narrative(p, rank, total):
    parts = []
    parts.append("**The hottest property in your portfolio.**" if rank == 1
                 else f"**#{rank} of {total} hottest in your portfolio.**")
    pct_phrase = (f" (AOI {p.aoi_percentile:.0f}th percentile)"
                  if pd.notna(p.aoi_percentile) else "")
    parts.append(f"Surface temperature peaks at **{p.peak_temp_c:.1f} °C** at "
                 f"{int(p.peak_hour):02d}:00{pct_phrase}.")
    if pd.notna(p.impervious_pct) and pd.notna(p.vegetation_pct):
        parts.append(
            f"The surrounding scene is **{p.impervious_pct:.0f}% paved or built "
            f"and only {p.vegetation_pct:.0f}% vegetation** — that imbalance is "
            f"the primary heat driver."
        )
    if pd.notna(p.peak_heat_index_c) and p.peak_heat_index_c >= NOAA_EXTREME_CAUTION_C:
        hours = int(p.hours_above_sla) if pd.notna(p.hours_above_sla) else 0
        parts.append(
            f"Heat index reaches **{p.peak_heat_index_c:.1f} °C** — above NOAA's "
            f"Extreme Caution threshold — for {hours} business hours."
        )
    return " ".join(parts)

def _md_to_html(s):
    """Tiny markdown-bold → HTML conversion for the narrative."""
    out, bold = [], False
    i = 0
    while i < len(s):
        if s[i:i+2] == '**':
            out.append('</b>' if bold else '<b>')
            bold = not bold
            i += 2
        else:
            out.append(s[i])
            i += 1
    return ''.join(out)

top3 = top_n.sort_values('temp_rank').reset_index(drop=True)
total = len(top3)

print(f"Action brief — top {total} hottest properties.\n"
      "Every recommendation cites a public intervention program.\n")

for _, p in top3.iterrows():
    rank = int(p.temp_rank)
    action_label, action_why, action_cite = _pick_action(p)
    narrative_html = _md_to_html(_narrative(p, rank, total))

    # Explicit colors on every text node so the card stays readable in both
    # light and dark Jupyter / VS Code themes (themes can override inherited
    # text color but rarely override inline `color:` attributes).
    display(HTML(f'''
    <div style="border:1px solid #ccc;border-radius:6px;padding:18px 20px;margin:12px 0;
                font:13px/1.6 -apple-system,sans-serif;background:#ffffff;color:#1a1a1a">
      <div style="font:600 15px sans-serif;margin-bottom:10px;color:#0d0d0d">
        #{rank} · {p.property_id} — {p['name']}
        <span style="color:#555;font-weight:400">
          · {p['type']} · {int(p['sqft']):,} sqft · ${p['market_value_musd']}M
        </span>
      </div>
      <div style="margin:6px 0;color:#1a1a1a">{narrative_html}</div>
      <div style="margin-top:14px;padding:12px 16px;border-left:4px solid #d73027;
                  background:#fff5f5;border-radius:0 4px 4px 0;color:#1a1a1a">
        <div style="font:600 14px sans-serif;color:#0d0d0d">→ {action_label}</div>
        <div style="margin:4px 0;color:#333">{action_why.capitalize()}.</div>
        <div style="font-size:11px;color:#666;margin-top:6px">
          Reference: {action_cite}
        </div>
      </div>
    </div>
    '''))

# ── Final ranked map — top-N markers tooltip-labeled with the action ─────────
m3_lats = top3['latitude'].tolist()
m3_lons = top3['longitude'].tolist()
m3_center = ([sum(m3_lats) / len(m3_lats), sum(m3_lons) / len(m3_lons)]
             if m3_lats else center)
m3 = folium.Map(location=m3_center, zoom_start=14, tiles='cartodbpositron')

# Underlay — the heatmap tile each property sits on, faintly tinted.
seen_tiles = set()
for _, p in top3.iterrows():
    t = tile_for(p.latitude, p.longitude)
    if t is None:
        continue
    poly, _, peak_c, peak_h, _ = t
    key = (round(poly.centroid.x, 6), round(poly.centroid.y, 6))
    if key in seen_tiles:
        continue
    seen_tiles.add(key)
    folium.GeoJson(
        mapping(poly),
        style_function=lambda x: {
            'fillColor': '#d73027', 'color': '#d73027',
            'weight': 0.6, 'fillOpacity': 0.25,
        },
        tooltip=f"{peak_c:.1f}°C peak @ {peak_h:02d}:00",
    ).add_to(m3)

# Markers — sized & colored by measured peak temperature; popup shows the action.
peak_lo = float(top3['peak_temp_c'].min())
peak_hi = float(top3['peak_temp_c'].max())
for _, p in top3.iterrows():
    action_label, _, action_cite = _pick_action(p)
    folium.CircleMarker(
        location=[p.latitude, p.longitude],
        radius=10 + (p.peak_temp_c - peak_lo) * 4,
        color='black', weight=1,
        fill=True,
        fill_color=temp_color(p.peak_temp_c, peak_lo, peak_hi),
        fill_opacity=0.92,
        tooltip=f"#{int(p.temp_rank)} {p['property_id']} — {action_label}",
        popup=(f"<b>#{int(p.temp_rank)} {p['name']}</b><br/>"
               f"peak: {p.peak_temp_c:.1f}°C @ {int(p.peak_hour):02d}:00<br/>"
               f"<b>→ {action_label}</b><br/>"
               f"<span style='color:#666;font-size:11px'>{action_cite}</span>"),
    ).add_to(m3)

if m3_lats:
    pad_lat = max(0.004, (max(m3_lats) - min(m3_lats)) * 0.30)
    pad_lon = max(0.004, (max(m3_lons) - min(m3_lons)) * 0.30)
    m3.fit_bounds([[min(m3_lats) - pad_lat, min(m3_lons) - pad_lon],
                   [max(m3_lats) + pad_lat, max(m3_lons) + pad_lon]])

m3.get_root().html.add_child(folium.Element(
    '<div style="position:fixed;bottom:30px;left:30px;z-index:9999;'
    'background:white;padding:8px 12px;border:1px solid #888;font:12px sans-serif;">'
    f'<b>Top-{total} hottest · recommended actions</b><br/>'
    '<span style="color:#666;">marker color &amp; size ∝ measured peak temperature</span><br/>'
    '<span style="color:#666;">click a marker for the recommendation</span>'
    '</div>'
))
display(m3)

---
## Wrap-up

Starting from a real-estate portfolio CSV you now have evidence-based answers for the client meeting:

| Artifact | Used by |
|----------|---------|
| **M1** portfolio overview map | Client deck — slide 1 |
| **M2** hot-exposures zoom map | Client deck — slide 2 |
| **M3** final ranked priority map | Client deck — slide 3 |
| **C1** diurnal temperature curves | Backup / analyst review |
| **C2** surface composition stacked bar | Backup |
| **C3** HI / apparent / RH / solar profile (#1) | Deep-dive slide |
| **C4** risk vs opportunity scatter (4 quadrants) | Decision slide |
| `outputs/portfolio_evaluation.csv` | CRM / client report |
| [Heat-intelligence PDF report](../../data/real_state_san_jose_heat_intelligence_sample_day_2024-10-02.pdf) | Forward to client unchanged |

Every input is explicit. Every weight is at the top of the notebook. Set `CACHED=False` to rerun against any portfolio with a live FortyGuard API key — the same workflow applies, with the caveat that diurnal columns reduce to the snapshot hour until a 24-hour live API is available.

**Apply this pattern to adjacent use cases**: insured-properties portfolio (insurance underwriting), data-center sites (operational-risk screening), hospitality assets (guest-comfort benchmarking), retail acquisitions (foot-traffic comfort). The workflow — *portfolio × diurnal heatmap × surface diagnosis × ground-truth × env-params → risk + opportunity table* — transfers directly.